# Governance Role Clustering Framework

This notebook builds a governance-focused role clustering framework using cybersecurity job data. It applies NLP preprocessing, TF-IDF vectorization, K-means clustering, silhouette analysis, and hierarchical clustering visualization to identify natural governance role domains from cybersecurity job-role text.

## Business Problem

Cybersecurity organizations often need to organize responsibilities across role domains such as audit, compliance, risk, security operations, and governance. Job descriptions contain signals about these responsibilities, but they are usually unstructured text.

This notebook treats job-role text as analytical data and uses clustering to identify patterns that can support role grouping and workforce planning.

In [ ]:
# Core libraries
import os
import re
import glob
import string
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# NLP
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer

# Machine learning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Hierarchical clustering
from scipy.cluster.hierarchy import linkage, dendrogram

# Kaggle dataset download
import kagglehub

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Download NLTK resources
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

## Load Cybersecurity Job Data

In [ ]:
# Download the Kaggle dataset by Danny Revaldo
# Dataset: Salary Cyber Security Jobs
dataset_path = kagglehub.dataset_download("dannyrevaldo/salary-cyber-security-jobs")
print("Dataset downloaded to:", dataset_path)

# Find CSV files inside the downloaded dataset folder
csv_files = glob.glob(os.path.join(dataset_path, "**", "*.csv"), recursive=True)
print("CSV files found:")
for file in csv_files:
    print("-", file)

if len(csv_files) == 0:
    raise FileNotFoundError("No CSV file was found in the Kaggle dataset folder.")

df = pd.read_csv(csv_files[0])

print("Loaded file:", csv_files[0])
display(df.head())
print("Dataset shape:", df.shape)
print("Columns:", list(df.columns))

## Build Role Text Field

In [ ]:
# Identify text-like columns that can contribute to role clustering
object_columns = df.select_dtypes(include=["object"]).columns.tolist()

print("Object/text-like columns detected:")
print(object_columns)

# Create a combined role_text field from all text-like columns.
df_work = df.copy()

for col in object_columns:
    df_work[col] = df_work[col].fillna("").astype(str)

df_work["role_text"] = df_work[object_columns].agg(" ".join, axis=1)
df_work["role_text"] = df_work["role_text"].str.replace(r"\s+", " ", regex=True).str.strip()
df_work = df_work[df_work["role_text"].str.len() > 0].reset_index(drop=True)
df_work["document_id"] = range(1, len(df_work) + 1)

print("Text column used for clustering: role_text")
print("Working dataset shape:", df_work.shape)
display(df_work[["document_id", "role_text"]].head())

## NLP Preprocessing

In [ ]:
custom_stop_words = {
    "job", "jobs", "role", "position", "company", "salary", "year", "years",
    "experience", "required", "preferred", "work", "working", "team",
    "candidate", "candidates", "responsibilities", "requirements", "skills",
    "ability", "knowledge", "including", "etc", "remote", "hybrid", "onsite",
    "full", "time", "part", "new", "posted", "apply"
}

english_stop_words = set(stopwords.words("english"))
stop_words = english_stop_words.union(custom_stop_words)
stemmer = SnowballStemmer("english")
punctuation_table = str.maketrans("", "", string.punctuation)

def tokenize_text(text):
    text = str(text).lower()
    text = text.replace("’", "'")
    return word_tokenize(text)

def remove_stopwords_and_punctuation(tokens):
    cleaned = []
    for token in tokens:
        token = token.translate(punctuation_table)
        token = re.sub(r"[^a-zA-Z]", "", token)
        if len(token) >= 3 and token not in stop_words:
            cleaned.append(token)
    return cleaned

def stem_tokens(tokens):
    return [stemmer.stem(token) for token in tokens]

df_work["tokens"] = df_work["role_text"].apply(tokenize_text)
df_work["tokens_no_stop"] = df_work["tokens"].apply(remove_stopwords_and_punctuation)
df_work["stemmed_tokens"] = df_work["tokens_no_stop"].apply(stem_tokens)
df_work["processed_text"] = df_work["stemmed_tokens"].apply(lambda tokens: " ".join(tokens))

display(df_work[["role_text", "tokens", "tokens_no_stop", "stemmed_tokens", "processed_text"]].head(3))

## TF-IDF Feature Engineering

In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    max_df=0.85,
    min_df=2,
    max_features=1000,
    ngram_range=(1, 2),
    token_pattern=r"(?u)\b\w\w\w+\b"
)

tfidf_matrix = tfidf_vectorizer.fit_transform(df_work["processed_text"])
terms = tfidf_vectorizer.get_feature_names_out()

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("\nSample extracted terms:")
print(terms[:30])

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=terms,
    index=df_work["document_id"]
)

tfidf_transposed = tfidf_df.T
print("Transposed TF-IDF DataFrame shape:", tfidf_transposed.shape)
display(tfidf_transposed.iloc[:20, :10])

## K-Means Governance Role Clustering

In [ ]:
initial_k = 5

kmeans_model = KMeans(
    n_clusters=initial_k,
    random_state=RANDOM_STATE,
    n_init=10
)

kmeans_model.fit(tfidf_matrix)
df_work["cluster"] = kmeans_model.labels_

cluster_assignments = df_work[["document_id", "cluster", "role_text"]].copy()
display(cluster_assignments.head(20))

print("Cluster distribution:")
cluster_distribution = df_work["cluster"].value_counts().sort_index()
display(cluster_distribution.to_frame("document_count"))

plt.figure(figsize=(8, 5))
cluster_distribution.plot(kind="bar")
plt.title("Distribution of Cybersecurity Job Records Across K-Means Clusters")
plt.xlabel("Cluster")
plt.ylabel("Number of Documents")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Representative Cluster Terms

In [ ]:
def get_top_terms_per_cluster(model, feature_names, top_n=15):
    order_centroids = model.cluster_centers_.argsort()[:, ::-1]
    cluster_terms = {}
    for cluster_id in range(model.n_clusters):
        top_terms = [feature_names[ind] for ind in order_centroids[cluster_id, :top_n]]
        cluster_terms[cluster_id] = top_terms
    return cluster_terms

top_terms = get_top_terms_per_cluster(kmeans_model, terms, top_n=15)

print("Top representative terms for each cluster:")
for cluster_id, cluster_terms in top_terms.items():
    print(f"\nCluster {cluster_id}:")
    print(", ".join(cluster_terms))

## Governance Domain Labeling

In [ ]:
def suggest_governance_label(cluster_terms):
    term_string = " ".join(cluster_terms).lower()
    scores = {
        "Audit": sum(word in term_string for word in ["audit", "control", "test", "evid", "review", "assur"]),
        "Compliance": sum(word in term_string for word in ["compli", "regul", "policy", "standard", "require"]),
        "Risk": sum(word in term_string for word in ["risk", "assess", "mitig", "threat", "vulnerab"]),
        "Security Operations": sum(word in term_string for word in ["secur", "incident", "monitor", "detect", "response", "oper"]),
        "Governance": sum(word in term_string for word in ["govern", "manag", "program", "strateg", "lead", "oversight"])
    }
    return max(scores, key=scores.get)

cluster_label_map = {cluster_id: suggest_governance_label(cluster_terms) for cluster_id, cluster_terms in top_terms.items()}
df_work["governance_domain"] = df_work["cluster"].map(cluster_label_map)

cluster_summary = pd.DataFrame({
    "cluster": list(top_terms.keys()),
    "suggested_governance_domain": [cluster_label_map[c] for c in top_terms.keys()],
    "top_terms": [", ".join(top_terms[c]) for c in top_terms.keys()],
    "document_count": [int((df_work["cluster"] == c).sum()) for c in top_terms.keys()]
})

display(cluster_summary)

print("Interpretation:")
print(
    "The K-means model groups cybersecurity job-role records into clusters based on similar TF-IDF term patterns. "
    "The governance domain labels translate those clusters into role-assignment categories: Audit, Compliance, "
    "Risk, Security Operations, and Governance. The labels should be evaluated against the top terms and sample "
    "job records in each cluster."
)

## Cluster Selection with Silhouette Analysis

In [ ]:
silhouette_results = []
max_k = min(20, tfidf_matrix.shape[0] - 1)

for k in range(2, max_k + 1):
    model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = model.fit_predict(tfidf_matrix)
    score = silhouette_score(tfidf_matrix, labels)
    silhouette_results.append({"k": k, "silhouette_score": score})

silhouette_df = pd.DataFrame(silhouette_results)
display(silhouette_df)

optimal_row = silhouette_df.loc[silhouette_df["silhouette_score"].idxmax()]
optimal_k = int(optimal_row["k"])
optimal_score = float(optimal_row["silhouette_score"])

print(f"Optimal K based on silhouette score: {optimal_k}")
print(f"Best silhouette score: {optimal_score:.4f}")

plt.figure(figsize=(10, 5))
plt.plot(silhouette_df["k"], silhouette_df["silhouette_score"], marker="o")
plt.title("Silhouette Score by Number of Clusters")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Silhouette score")
plt.xticks(silhouette_df["k"])
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Data-Driven Final Cluster View

In [ ]:
final_kmeans = KMeans(n_clusters=optimal_k, random_state=RANDOM_STATE, n_init=10)
df_work["optimal_k_cluster"] = final_kmeans.fit_predict(tfidf_matrix)
final_top_terms = get_top_terms_per_cluster(final_kmeans, terms, top_n=15)

print("Top terms for final silhouette-selected K-means model:")
for cluster_id, cluster_terms in final_top_terms.items():
    print(f"\nCluster {cluster_id}:")
    print(", ".join(cluster_terms))

display(df_work[["document_id", "cluster", "governance_domain", "optimal_k_cluster", "role_text"]].head(20))

## Hierarchical Governance Taxonomy

In [ ]:
sample_size = min(40, tfidf_matrix.shape[0])
sample_indices = np.random.choice(tfidf_matrix.shape[0], sample_size, replace=False)

sample_matrix = tfidf_matrix[sample_indices].toarray()
sample_labels = (
    df_work.iloc[sample_indices]["document_id"].astype(str)
    + " | "
    + df_work.iloc[sample_indices]["governance_domain"].astype(str)
)

linked = linkage(sample_matrix, method="ward")

plt.figure(figsize=(18, 8))
dendrogram(
    linked,
    labels=sample_labels.values,
    leaf_rotation=90,
    leaf_font_size=8
)
plt.title("Hierarchical Clustering Dendrogram of Cybersecurity Job Role Records")
plt.xlabel("Document ID | Governance Domain")
plt.ylabel("Operational/Textual Dissimilarity")
plt.tight_layout()
plt.show()

print(
    "The dendrogram supports the role-assignment theme by showing how individual job-role records can be merged "
    "into broader governance groupings. Earlier merges suggest stronger similarity; later merges suggest broader "
    "and less similar governance categories."
)

## Reinforcement Learning Extension: Governance Learning Policy

This extension compares two Q-learning agents trained with different learning rates. Learning rate is interpreted as the speed at which a role-assignment process updates after feedback.

- **Simon:** conservative learning policy
- **Olive:** aggressive learning policy

In [ ]:
from game import MemoryTicTacToe, TicTacToe
from bots import RLBot, RandomBot

print("Imported MemoryTicTacToe, TicTacToe, RLBot, and RandomBot successfully.")

In [ ]:
def board_is_full(game):
    return all(cell != " " for row in game.board for cell in row)

def reward_for_outcome(winner, bot_player):
    if winner == bot_player:
        return 1
    elif winner == " ":
        return 0
    else:
        return -1

def train_bot_against_random(bot_name, learning_rate, episodes=5000, discount_factor=0.9, exploration=0.1):
    training_log = []
    game = MemoryTicTacToe()
    rl_bot = RLBot(game, player="O", learning_rate=learning_rate, discount_factor=discount_factor, exploration=exploration)

    for episode in range(episodes):
        game.start_over()
        random_bot = RandomBot(game, player="X")
        rl_bot.change_game(game)

        bot_history = []

        while game.winner == " " and not board_is_full(game):
            if game.player == "X":
                random_bot.move()
            else:
                old_state, action, new_state = rl_bot.move()
                bot_history.append((old_state, action, new_state))

        outcome_reward = reward_for_outcome(game.winner, rl_bot.player)

        for old_state, action, new_state in bot_history:
            rl_bot.update_q_values(old_state, action, outcome_reward, new_state)

        training_log.append({
            "episode": episode + 1,
            "bot": bot_name,
            "learning_rate": learning_rate,
            "winner": game.winner if game.winner != " " else "draw",
            "reward": outcome_reward,
            "moves_by_bot": len(bot_history)
        })

    return rl_bot, pd.DataFrame(training_log)

simon_bot, simon_log = train_bot_against_random(
    bot_name="Simon",
    learning_rate=0.10,
    episodes=5000,
    discount_factor=0.9,
    exploration=0.1
)

olive_bot, olive_log = train_bot_against_random(
    bot_name="Olive",
    learning_rate=0.70,
    episodes=5000,
    discount_factor=0.9,
    exploration=0.1
)

training_results = pd.concat([simon_log, olive_log], ignore_index=True)
display(training_results.head())
display(training_results.groupby(["bot", "winner"]).size().unstack(fill_value=0))

In [ ]:
def evaluate_trained_bot(bot, bot_name, games=500):
    old_exploration = bot.exploration_rate
    bot.exploration_rate = 0.0

    results = []
    for game_number in range(games):
        game = MemoryTicTacToe()
        game.start_over()

        bot.change_game(game)
        random_bot = RandomBot(game, player="X")

        moves = 0
        while game.winner == " " and not board_is_full(game):
            if game.player == "X":
                random_bot.move()
            else:
                bot.move()
            moves += 1

        reward = reward_for_outcome(game.winner, bot.player)
        results.append({
            "game": game_number + 1,
            "bot": bot_name,
            "winner": game.winner if game.winner != " " else "draw",
            "reward": reward,
            "moves": moves
        })

    bot.exploration_rate = old_exploration
    return pd.DataFrame(results)

simon_eval = evaluate_trained_bot(simon_bot, "Simon", games=500)
olive_eval = evaluate_trained_bot(olive_bot, "Olive", games=500)

evaluation_results = pd.concat([simon_eval, olive_eval], ignore_index=True)

performance_summary = evaluation_results.groupby("bot").agg(
    average_reward=("reward", "mean"),
    average_moves=("moves", "mean"),
    wins=("winner", lambda x: (x == "O").sum()),
    losses=("winner", lambda x: (x == "X").sum()),
    draws=("winner", lambda x: (x == "draw").sum())
).reset_index()

display(performance_summary)

training_results["rolling_reward"] = (
    training_results.groupby("bot")["reward"]
    .transform(lambda s: s.rolling(window=100, min_periods=1).mean())
)

plt.figure(figsize=(10, 5))
for bot_name in training_results["bot"].unique():
    subset = training_results[training_results["bot"] == bot_name]
    plt.plot(subset["episode"], subset["rolling_reward"], label=bot_name)

plt.title("Training Reward Trend by Learning Rate")
plt.xlabel("Training Episode")
plt.ylabel("Rolling Average Reward")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Optional Interactive Play: Simon

In [ ]:
simon_game = TicTacToe()
simon_bot.change_game(simon_game)
simon_game.set_bot(simon_bot)
simon_game.display()

## Optional Interactive Play: Olive

In [ ]:
olive_game = TicTacToe()
olive_bot.change_game(olive_game)
olive_game.set_bot(olive_bot)
olive_game.display()

## Learning Rate Interpretation

In [ ]:
print("Learning Rate Interpretation")
print("----------------------------")
print(
    "Simon used a lower learning rate (0.10), which means he updated Q-values more conservatively after each outcome. "
    "This resembles a governance role-assignment process that changes slowly after feedback."
)
print(
    "Olive used a higher learning rate (0.70), which means she updated Q-values more aggressively after each outcome. "
    "This resembles a governance role-assignment process that adapts quickly when feedback indicates a better or worse decision."
)
print(
    "The performance summary compares perceived intelligence through wins, losses, draws, reward, and average moves. "
    "If Olive improves faster but is less stable, the high learning rate made the agent more reactive. If Simon performs more steadily, "
    "the lower learning rate produced slower but more stable learning."
)

## Portfolio Conclusion

The notebook demonstrates how unsupervised learning can convert cybersecurity job-role text into interpretable governance role domains. The K-means model provides a flat role segmentation, silhouette analysis supports cluster selection, and the dendrogram provides a hierarchy for understanding broader role relationships.

This framework can be expanded into a governance workforce analysis tool by adding richer role descriptions, control ownership language, certification requirements, and organizational reporting structures.